In [21]:
from zk import ZK, const

conn = None
# create ZK instance
zk = ZK('10.254.26.251', port=4370, timeout=5, password=171, force_udp=False, ommit_ping=False)
try:
    # connect to device
    conn = zk.connect()
    # disable device, this method ensures no activity on the device while the process is run
    conn.disable_device()
    # another commands will be here!
    # Example: Get All Users
    users = conn.get_users()
    for user in users:
        privilege = 'User'
        if user.privilege == const.USER_ADMIN:
            privilege = 'Admin'
        print ('+ UID #{}'.format(user.uid))
        print ('  Name       : {}'.format(user.name))
        print ('  Privilege  : {}'.format(privilege))
        print ('  Password   : {}'.format(user.password))
        print ('  Group ID   : {}'.format(user.group_id))
        print ('  User  ID   : {}'.format(user.user_id))

    # Test Voice: Say Thank You
    #conn.test_voice()
    # re-enable device after all commands already executed
    conn.enable_device()
except Exception as e:
    print ("Process terminate : {}".format(e))
finally:
    if conn:
        conn.disconnect()

+ UID #1
  Name       : admin
  Privilege  : Admin
  Password   : 0000
  Group ID   : 
  User  ID   : 1
+ UID #3
  Name       : DavidRR
  Privilege  : User
  Password   : 1234
  Group ID   : 
  User  ID   : 2
+ UID #4
  Name       : JoseSaRC
  Privilege  : User
  Password   : 1234
  Group ID   : 
  User  ID   : 3
+ UID #5
  Name       : ArmandoGC
  Privilege  : User
  Password   : 
  Group ID   : 
  User  ID   : 4
+ UID #6
  Name       : yo
  Privilege  : User
  Password   : 
  Group ID   : 
  User  ID   : 5
+ UID #7
  Name       : el
  Privilege  : User
  Password   : 
  Group ID   : 
  User  ID   : 6
+ UID #8
  Name       : Mike
  Privilege  : User
  Password   : 1212
  Group ID   : 
  User  ID   : 1000
+ UID #9
  Name       : Chiquilla
  Privilege  : User
  Password   : 
  Group ID   : 
  User  ID   : 7


In [7]:
from zk import ZK
from datetime import datetime
from collections import defaultdict

IP_RELOJ = "10.254.26.251"
PUERTO = 4370
PASSWORD = 171  

USER_ID_OBJETIVO = "5"

ANIO = 2026
MES = 5

FECHA_INICIO = datetime(ANIO, MES, 1)
FECHA_FIN = datetime(ANIO, MES + 1, 1)

zk = ZK(
    IP_RELOJ,
    port=PUERTO,
    timeout=20,
    password=PASSWORD,
    force_udp=False,
    ommit_ping=False
)

conn = None

try:
    conn = zk.connect()
    attendances = conn.get_attendance()

    registros_por_dia = defaultdict(list)

    for att in attendances:
        if str(att.user_id) == USER_ID_OBJETIVO and FECHA_INICIO <= att.timestamp < FECHA_FIN:
            registros_por_dia[att.timestamp.date()].append(att)

    print(f"\nResumen de asistencia del User ID {USER_ID_OBJETIVO} - Mayo {ANIO}")
    print("=" * 90)
    print(f"{'Fecha':<12} {'Entrada':<10} {'Salida':<10} {'Checadas':<10} {'Horas':<10}")
    print("-" * 90)

    for fecha, registros in sorted(registros_por_dia.items()):
        registros.sort(key=lambda x: x.timestamp)

        entrada = registros[0].timestamp
        salida = registros[-1].timestamp

        duracion_horas = (salida - entrada).total_seconds() / 3600

        print(
            f"{str(fecha):<12} "
            f"{entrada.strftime('%H:%M:%S'):<10} "
            f"{salida.strftime('%H:%M:%S'):<10} "
            f"{len(registros):<10} "
            f"{duracion_horas:<10.2f}"
        )

finally:
    if conn:
        conn.disconnect()

ZKNetworkError: timed out

In [1]:
from zk import ZK, const

conn = None
# create ZK instance
zk = ZK('10.254.26.251', port=4370, timeout=5, password=171, force_udp=False, ommit_ping=False)
try:
    # connect to device
    conn = zk.connect()
    # disable device, this method ensures no activity on the device while the process is run
    conn.disable_device()
    # another commands will be here!
    # Example: Get All Users
    users = conn.get_users()
    for user in users:
        privilege = 'User'
        if user.privilege == const.USER_ADMIN:
            privilege = 'Admin'
        print ('+ UID #{}'.format(user.uid))
        print ('  Name       : {}'.format(user.name))
        print ('  Privilege  : {}'.format(privilege))
        print ('  Password   : {}'.format(user.password))
        print ('  Group ID   : {}'.format(user.group_id))
        print ('  User  ID   : {}'.format(user.user_id))

    # Test Voice: Say Thank You
    #conn.test_voice()
    # re-enable device after all commands already executed
    voces = {
        0: "Thank You",
        1: "Incorrect Password",
        2: "Access Denied",
        3: "Invalid ID",
        4: "Please try again",
        5: "Duplicate ID",
        6: "The clock is full",
        8: "Duplicate finger",
        9: "Duplicated punch",
        10: "Beep kuko",
        11: "Beep siren",
        13: "Beep bell",
        18: "Windows opening sound",
        24: "Beep standard",
        30: "Invalid user",
        33: "Illegal Access",
        34: "Disk space full",
        35: "Duplicate fingerprint",
        36: "Fingerprint not registered",
        51: "Focus eyes on the green box"
    }
    conn.test_voice(index=0)
    #conn.enable_device()
except Exception as e:
    print ("Process terminate : {}".format(e))
finally:
    if conn:
        conn.disconnect()

Process terminate : timed out


In [5]:
from zk import ZK, const
from contextlib import suppress


ZK_IP = "10.254.26.251"
ZK_PORT = 4370
ZK_PASSWORD = 171


def clean_text(value):
    """
    Limpia textos recibidos del reloj.
    Evita caracteres raros o espacios innecesarios.
    """
    if value is None:
        return ""

    text = str(value).strip()
    text = "".join(ch for ch in text if ch.isprintable())

    return text.strip()


def normalize_group_id(value):
    """
    Algunos relojes devuelven el grupo como carácter de control.
    Por ejemplo: '\x01' puede representar grupo 1.
    """
    if value is None:
        return ""

    text = str(value)

    if len(text) == 1 and ord(text) < 32:
        return str(ord(text))

    return clean_text(text)


def get_privilege_label(privilege):
    if privilege == const.USER_ADMIN:
        return "Admin"
    return "User"


def main():
    conn = None
    device_disabled = False

    try:
        zk = ZK(
            ZK_IP,
            port=ZK_PORT,
            timeout=5,
            password=ZK_PASSWORD,
            force_udp=False,
            ommit_ping=False
        )

        print("Conectando al reloj...")
        conn = zk.connect()

        print("Conexión exitosa.")
        print("Deshabilitando temporalmente el dispositivo para lectura segura...")
        conn.disable_device()
        device_disabled = True

        users = conn.get_users()

        print(f"\nUsuarios encontrados en el reloj: {len(users)}\n")

        for user in users:
            zk_uid = user.uid
            zk_user_id = clean_text(user.user_id)
            name = clean_text(user.name)
            privilege = get_privilege_label(user.privilege)
            group_id = normalize_group_id(user.group_id)
            tiene_pin = bool(clean_text(user.password))

            print(f"+ UID #{zk_uid}")
            print(f"  Name       : {name}")
            print(f"  Privilege  : {privilege}")
            print(f"  Has PIN    : {'Sí' if tiene_pin else 'No'}")
            print(f"  Group ID   : {group_id}")
            print(f"  User ID    : {zk_user_id}")
            print("-" * 40)

        # Prueba opcional de voz
        conn.test_voice(index=0)

    except Exception as e:
        print(f"Process terminate: {e}")

    finally:
        if conn:
            if device_disabled:
                print("Rehabilitando dispositivo...")
                with suppress(Exception):
                    conn.enable_device()

            print("Cerrando conexión...")
            with suppress(Exception):
              conn.disconnect()


if __name__ == "__main__":
    main()

Conectando al reloj...
Conexión exitosa.
Deshabilitando temporalmente el dispositivo para lectura segura...

Usuarios encontrados en el reloj: 10

+ UID #1
  Name       : admin
  Privilege  : Admin
  Has PIN    : Sí
  Group ID   : 
  User ID    : 1
----------------------------------------
+ UID #3
  Name       : DavidRR
  Privilege  : User
  Has PIN    : Sí
  Group ID   : 1
  User ID    : 2
----------------------------------------
+ UID #4
  Name       : JoseSaRC
  Privilege  : User
  Has PIN    : Sí
  Group ID   : 
  User ID    : 3
----------------------------------------
+ UID #5
  Name       : ArmandoGC
  Privilege  : User
  Has PIN    : No
  Group ID   : 
  User ID    : 4
----------------------------------------
+ UID #6
  Name       : yo
  Privilege  : User
  Has PIN    : No
  Group ID   : 
  User ID    : 5
----------------------------------------
+ UID #7
  Name       : el
  Privilege  : User
  Has PIN    : No
  Group ID   : 
  User ID    : 6
-------------------------------------

In [1]:
from zk import ZK, const
from contextlib import suppress
import random


ZK_IP = "10.254.26.251"
ZK_PORT = 4370
ZK_PASSWORD = 171


def clean_text(value):
    if value is None:
        return ""

    text = str(value).strip()
    text = "".join(ch for ch in text if ch.isprintable())
    return text.strip()


def get_next_uid(users):
    """
    Obtiene el siguiente UID interno disponible.
    El UID es el identificador interno del reloj.
    """
    existing_uids = [int(user.uid) for user in users if user.uid is not None]

    if not existing_uids:
        return 1

    return max(existing_uids) + 1


def generate_random_user_id(users):
    """
    Genera un User ID aleatorio que no exista en el reloj.
    Este es el ID visible del usuario en ZKTeco.
    """
    existing_user_ids = {
        clean_text(user.user_id)
        for user in users
        if clean_text(user.user_id)
    }

    for _ in range(100):
        candidate = str(random.randint(9000, 9999))

        if candidate not in existing_user_ids:
            return candidate

    raise Exception("No se pudo generar un user_id disponible.")


def print_users(users):
    print("\nUsuarios actuales en el reloj:\n")

    for user in users:
        privilege = "Admin" if user.privilege == const.USER_ADMIN else "User"

        print(f"+ UID #{user.uid}")
        print(f"  Name      : {clean_text(user.name)}")
        print(f"  User ID   : {clean_text(user.user_id)}")
        print(f"  Privilege : {privilege}")
        print("-" * 40)


def main():
    conn = None
    device_disabled = False

    try:
        zk = ZK(
            ZK_IP,
            port=ZK_PORT,
            timeout=5,
            password=ZK_PASSWORD,
            force_udp=False,
            ommit_ping=False
        )

        print("Conectando al reloj...")
        conn = zk.connect()

        print("Conexión exitosa.")
        conn.disable_device()
        device_disabled = True

        users_before = conn.get_users()

        print_users(users_before)

        new_uid = get_next_uid(users_before)
        new_user_id = generate_random_user_id(users_before)

        new_name = "Puebita de hoy"
        new_password = "1234"
        new_privilege = const.USER_DEFAULT
        new_group_id = ""

        print("\nRegistrando nuevo usuario de prueba...")
        print(f"  UID interno : {new_uid}")
        print(f"  User ID     : {new_user_id}")
        print(f"  Nombre      : {new_name}")
        print(f"  Password    : {new_password}")
        print(f"  Privilegio  : Usuario normal")

        conn.set_user(
            uid=new_uid,
            name=new_name,
            privilege=new_privilege,
            password=new_password,
            group_id=new_group_id,
            user_id=new_user_id
        )

        print("\nUsuario creado. Verificando lectura desde el reloj...")

        users_after = conn.get_users()

        created_user = None

        for user in users_after:
            if int(user.uid) == int(new_uid):
                created_user = user
                break

        if created_user:
            print("\nUsuario encontrado correctamente:\n")
            print(f"+ UID #{created_user.uid}")
            print(f"  Name      : {clean_text(created_user.name)}")
            print(f"  User ID   : {clean_text(created_user.user_id)}")
            print(f"  Privilege : {'Admin' if created_user.privilege == const.USER_ADMIN else 'User'}")
            print(f"  Password  : {clean_text(created_user.password)}")
        else:
            print("\nEl usuario se intentó crear, pero no apareció en la lectura posterior.")

    except Exception as e:
        print(f"Process terminate: {e}")

    finally:
        if conn:
            if device_disabled:
                print("\nRehabilitando dispositivo...")
                with suppress(Exception):
                    conn.enable_device()

            print("Cerrando conexión...")
            with suppress(Exception):
                conn.disconnect()


if __name__ == "__main__":
    main()

Conectando al reloj...
Conexión exitosa.

Usuarios actuales en el reloj:

+ UID #1
  Name      : admin
  User ID   : 1
  Privilege : Admin
----------------------------------------
+ UID #3
  Name      : DavidRR
  User ID   : 2
  Privilege : User
----------------------------------------
+ UID #4
  Name      : JoseSaRC
  User ID   : 3
  Privilege : User
----------------------------------------
+ UID #5
  Name      : ArmandoGC
  User ID   : 4
  Privilege : User
----------------------------------------
+ UID #6
  Name      : yo
  User ID   : 5
  Privilege : User
----------------------------------------
+ UID #7
  Name      : el
  User ID   : 6
  Privilege : User
----------------------------------------
+ UID #8
  Name      : Mike
  User ID   : 1000
  Privilege : User
----------------------------------------
+ UID #9
  Name      : Chiquilla
  User ID   : 7
  Privilege : User
----------------------------------------
+ UID #10
  Name      : Puebita de hoy
  User ID   : 9821
  Privilege : User

In [19]:
from zk import ZK, const
from contextlib import suppress


ZK_IP = "10.254.26.251"
ZK_PORT = 4370
ZK_PASSWORD = 171


TARGET_NAME = "Puebita de hoy"

TARGET_USER_IDS = {
    "9657",

}


def clean_text(value):
    if value is None:
        return ""

    text = str(value).strip()
    text = "".join(ch for ch in text if ch.isprintable())
    return text.strip()


def get_privilege_label(privilege):
    if privilege == const.USER_ADMIN:
        return "Admin"
    return "User"


def print_users(users, title):
    print(f"\n{title}\n")

    for user in users:
        print(f"+ UID #{user.uid}")
        print(f"  Name      : {clean_text(user.name)}")
        print(f"  User ID   : {clean_text(user.user_id)}")
        print(f"  Privilege : {get_privilege_label(user.privilege)}")
        print("-" * 40)


def find_users_to_delete(users):
    """
    Busca únicamente usuarios que cumplan las dos condiciones:
    1. Nombre exacto: Puebita de hoy
    2. User ID dentro de TARGET_USER_IDS

    Esto evita borrar usuarios reales por accidente.
    """
    matches = []

    for user in users:
        name = clean_text(user.name)
        user_id = clean_text(user.user_id)
        privilege = get_privilege_label(user.privilege)

        if name == TARGET_NAME and user_id in TARGET_USER_IDS and privilege != "Admin":
            matches.append(user)

    return matches


def main():
    conn = None
    device_disabled = False

    try:
        zk = ZK(
            ZK_IP,
            port=ZK_PORT,
            timeout=5,
            password=ZK_PASSWORD,
            force_udp=False,
            ommit_ping=False
        )

        print("Conectando al reloj...")
        conn = zk.connect()

        print("Conexión exitosa.")
        print("Deshabilitando temporalmente el dispositivo...")
        conn.disable_device()
        device_disabled = True

        users_before = conn.get_users()
        print_users(users_before, "Usuarios antes de borrar:")

        users_to_delete = find_users_to_delete(users_before)

        if not users_to_delete:
            print("\nNo se encontraron usuarios de prueba para borrar.")
            return

        print("\nUsuarios que serán borrados:\n")

        for user in users_to_delete:
            print(f"+ UID #{user.uid}")
            print(f"  Name    : {clean_text(user.name)}")
            print(f"  User ID : {clean_text(user.user_id)}")
            print("-" * 40)

        print("\nBorrando usuarios de prueba...")

        for user in users_to_delete:
            uid = int(user.uid)
            user_id = clean_text(user.user_id)

            print(f"Borrando UID #{uid} | User ID {user_id} | {clean_text(user.name)}")

            result = conn.delete_user(uid=uid)

            print(f"Resultado delete_user(uid={uid}): {result}")

        print("\nVerificando usuarios después del borrado...")

        users_after = conn.get_users()
        print_users(users_after, "Usuarios después de borrar:")

        remaining = find_users_to_delete(users_after)

        if remaining:
            print("\nAdvertencia: algunos usuarios de prueba siguen existiendo:")
            for user in remaining:
                print(f"  UID #{user.uid} | User ID {clean_text(user.user_id)} | {clean_text(user.name)}")
        else:
            print("\nLimpieza completada. Ya no están los usuarios de prueba seleccionados.")

    except Exception as e:
        print(f"Process terminate: {e}")

    finally:
        if conn:
            if device_disabled:
                print("\nRehabilitando dispositivo...")
                with suppress(Exception):
                    conn.enable_device()

            print("Cerrando conexión...")
            with suppress(Exception):
                conn.disconnect()


if __name__ == "__main__":
    main()

Conectando al reloj...
Conexión exitosa.
Deshabilitando temporalmente el dispositivo...

Usuarios antes de borrar:

+ UID #1
  Name      : admin
  User ID   : 1
  Privilege : Admin
----------------------------------------
+ UID #3
  Name      : DavidRR
  User ID   : 2
  Privilege : User
----------------------------------------
+ UID #4
  Name      : JoseSaRC
  User ID   : 3
  Privilege : User
----------------------------------------
+ UID #5
  Name      : ArmandoGC
  User ID   : 4
  Privilege : User
----------------------------------------
+ UID #6
  Name      : yo
  User ID   : 5
  Privilege : User
----------------------------------------
+ UID #7
  Name      : el
  User ID   : 6
  Privilege : User
----------------------------------------
+ UID #8
  Name      : Mike
  User ID   : 1000
  Privilege : User
----------------------------------------
+ UID #9
  Name      : Chiquilla
  User ID   : 7
  Privilege : User
----------------------------------------
+ UID #10
  Name      : Puebita de